# Center of Pressure (COP) Analysis for Postural Control

This notebook computes center-of-pressure (COP) sway metrics and a 95% confidence ellipse from force-plate data, and provides comparison-plot helpers for visualizing balance performance across conditions and age groups.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research on postural control in adults and children before and after balance-training interventions (Rocker Board and Wobble Board).

**What is new in this notebook compared to my earlier work**:
- Refactored into reusable functions rather than a flat sequential script
- Applies eigenvalue decomposition of the COP covariance matrix to derive the 95% confidence ellipse (chi-square based)
- Normalizes sway metrics by foot length for cross-age comparison
- Includes publication-style comparison plots across conditions and groups

---


## 1. Setup and Data Import

Load packages and read the trial. The CSV is a Vicon Nexus export with a semicolon delimiter, similar to the format used in my overground gait notebook, but here the section of interest is the **Devices** block (force-plate data) rather than the Trajectories block.

The trial naming convention is `{Subject} Trial {N}.csv` (e.g., `S01 Trial 2.csv`).

In [ ]:
# Import necessary packages

import pandas as pd
import numpy as np
import os,sys
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from scipy.interpolate import interp1d

In [ ]:
# Get Subject
Subject = input("Subject: ")

In [ ]:
# Change directory if necessary

Path = input("Path: ") 
newdirectory = Path + Subject
os.chdir(newdirectory)

# Check the changed Directory
os.getcwd()

In [ ]:
# Get filename
Trial = input("Trial: ")
filename = Subject + " " + "Trial" + " " + Trial

Subject foot length is used to normalize sway metrics across body sizes. This is essential when comparing adults and children directly, since absolute COP excursion scales with foot size.

In [ ]:
foot_length_m = 0.17

In [ ]:
# CSV file import as dataframe

#df = pd.read_csv(filename + '.csv')
df = pd.read_csv(filename + '.csv', delimiter=';')

df.head()

## 2. Locate the Devices Section

Vicon exports stack multiple sections in one CSV (Devices, Joints, Model Outputs, Trajectories). For COP analysis I need the Devices block, which contains the force-plate `FP2 - CoP` columns. As in my overground notebook, I scan for section labels rather than hard-coding row offsets, so the parser does not break when the export configuration changes.

In [ ]:
# Use Only When the delimiter is ';'
# Split the dataframe by ','

df_split = df['Devices'].str.split(',', expand=True)
df_split

In [ ]:
# Use when event data does not exsit
Device_idx = -1

for idx in df_split.index:
    if df_split.iloc[idx, 0] == "Trajectories" :
         TRJ_idx = idx

Device_idx, TRJ_idx

## 3. Extract COP Data

Slice the Devices block to get the force-plate COP columns. The relevant force plate here is `FP2`, and the COP values come in two channels (X = anterior-posterior, Y = medial-lateral). Vicon exports COP in millimeters, so I convert to centimeters by dividing by 10 — this keeps the downstream sway and ellipse metrics in cm and cm² respectively.

In [ ]:
# Get index for COP data 

Device_df = df_split.iloc[Device_idx + 3 : TRJ_idx,:]
Device_df.columns = df_split.iloc[Device_idx+2,:]
FP2_COP_idx = Device_df.columns.get_loc('FP2 - CoP')

# Create COP Dataframe

COP_df = Device_df.iloc[:, FP2_COP_idx : FP2_COP_idx + 2]
COP_df = COP_df.reset_index(drop=True)
COP_df = COP_df.drop([0, 1])
COP_df.columns = ['COPx', 'COPy']

# COP Dataframe data type change

col_names = ['COPx', 'COPy']
for col in col_names:
    COP_df[col] = pd.to_numeric(COP_df[col], errors="coerce") 
    
COP_df.dtypes

In [ ]:
# COP unit change mm to cm

COP_df = COP_df/10
COP_df

Convert the COP dataframe columns and time index to numpy arrays for the downstream numerical work.

In [ ]:
# Series to Numpy array

COPx = COP_df['COPx'].to_numpy()
COPy = COP_df['COPy'].to_numpy()
timestamps = COP_df.index.to_numpy()


## 4. COP Sway and Confidence Ellipse Calculation

### Reasoning

A widely used way to summarize postural sway is the **95% confidence ellipse** of the COP point cloud. The method:

1. Stack the COPx and COPy samples and compute their covariance matrix
2. Eigendecompose the covariance matrix — eigenvectors give the principal axes of sway, eigenvalues give the variance along each axis
3. Scale the eigenvalues by the chi-square value at 95% confidence (df=2 for bivariate data) to convert from variance to a 95%-coverage ellipse

This gives an ellipse whose **area, axis lengths, and rotation angle** summarize the spatial spread and dominant direction of postural sway during the trial.

In addition to the ellipse, I compute classical sway metrics:

- **Sway velocity** (AP and ML): the total path length divided by trial duration, computed separately for each axis
- **Sway range** (AP and ML): the peak-to-peak excursion of COP along each axis

### Normalization

All ellipse and sway metrics are also reported normalized by foot length (or foot length squared, for area). Without this normalization, children's smaller feet would appear to produce different sway just because of body-size scaling. Normalization makes the cross-age comparison interpretable.

In [ ]:
# Calculate COP sway variables and normalized value (by foot length)


def calculate_COP_ellipse(COPx: np.ndarray, COPy: np.ndarray, confidence_level: float = 0.95):
    data = np.vstack((COPx, COPy))
    mean_x, mean_y = np.mean(COPx), np.mean(COPy)
    cov = np.cov(data)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    chi2_val = chi2.ppf(confidence_level, df=2)
    width = 2 * np.sqrt(eigenvalues[0] * chi2_val)
    height = 2 * np.sqrt(eigenvalues[1] * chi2_val)
    area = np.pi * (width / 2) * (height / 2)  # Ellipse area
    return (mean_x, mean_y), width, height, angle, area

def calculate_sway_metrics(COPx: np.ndarray, COPy: np.ndarray, timestamps: np.ndarray):
    if not (len(COPx) == len(COPy) == len(timestamps)):
        raise ValueError("COPx, COPy, and timestamps must have the same length.")
    
    time_duration = timestamps[-1] - timestamps[0]
    if time_duration <= 0:
        raise ValueError("Timestamps must be increasing and cover positive duration.")

    COPx_diff = np.diff(COPx)
    COPy_diff = np.diff(COPy)
    total_distance_AP = np.sum(np.abs(COPx_diff))
    total_distance_ML = np.sum(np.abs(COPy_diff))
    sway_velocity_AP = total_distance_AP / time_duration
    sway_velocity_ML = total_distance_ML / time_duration
    sway_range_AP = np.max(COPx) - np.min(COPx)
    sway_range_ML = np.max(COPy) - np.min(COPy)

    return {
        "sway_velocity_AP (m/s)": sway_velocity_AP,
        "sway_velocity_ML (m/s)": sway_velocity_ML,
        "sway_range_AP (m)": sway_range_AP,
        "sway_range_ML (m)": sway_range_ML
    }

def compile_sway_results(COPx: np.ndarray, COPy: np.ndarray, timestamps: np.ndarray, foot_length: float):
    center, width, height, angle, ellipse_area = calculate_COP_ellipse(COPx, COPy)
    sway_metrics = calculate_sway_metrics(COPx, COPy, timestamps)

    # Normalize where appropriate
    results = {
        "ellipse_area_95% (cm^²)": ellipse_area,
        "ellipse_center": center,
        "ellipse_area_95% normalized (unitless)": ellipse_area / (foot_length ** 2),
        "ellipse_width (cm)": width,
        "ellipse_width normalized (unitless)": width / foot_length,
        "ellipse_height (cm)": height,
        "ellipse_height normalized (unitless)": height / foot_length,
        "ellipse_angle (deg)": angle,
        "sway_velocity_AP (m/s)": sway_metrics["sway_velocity_AP (m/s)"],
        "sway_velocity_ML (m/s)": sway_metrics["sway_velocity_ML (m/s)"],
        "sway_range_AP (cm)": sway_metrics["sway_range_AP (m)"],
        "sway_range_AP normalized (unitless)": sway_metrics["sway_range_AP (m)"] / foot_length,
        "sway_range_ML (cm)": sway_metrics["sway_range_ML (m)"],
        "sway_range_ML normalized (unitless)": sway_metrics["sway_range_ML (m)"] / foot_length,
    }

    return pd.DataFrame([results])

# Get final results DataFrame
results_df = compile_sway_results(COPx, COPy, timestamps, foot_length=foot_length_m)
results_df

Run the ellipse and sway computation for this trial.

In [ ]:
center, width, height, angle, ellipse_area = calculate_COP_ellipse(COPx, COPy)

## 5. Visualize the COP Path with the 95% Ellipse

Plot the raw COP trajectory together with the fitted 95% confidence ellipse. This is the per-trial sanity check: the ellipse should contain roughly 95% of the COP samples, and its orientation should align with the visually dominant direction of sway.

In [ ]:
def plot_COP_with_ellipse(COPx, COPy, center, width, height, angle):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(COPx, COPy, 'o', markersize=2, label="COP Path")
    ellipse = Ellipse(xy=center, width=width, height=height, angle=angle,
                      edgecolor='r', fc='None', lw=2, label="95% Ellipse")
    ax.add_patch(ellipse)
    ax.set_aspect('equal')
    ax.set_xlabel("COPx (cm)")
    ax.set_ylabel("COPy (cm)")
    ax.legend()
    plt.grid(True)
    plt.title("Center of Pressure (COP) with 95% Confidence Ellipse")
    plt.show()

plot_COP_with_ellipse(COPx, COPy, center, width, height, angle)

## 6. Save Per-Subject Results

Append this trial's results to a per-subject CSV so trials can be compared across the pre/post intervention conditions within a subject.

In [ ]:
# Save COP data in csv (by subject)

results_df['filename'] = filename  # add filename as a new column
cols = ['filename'] + [col for col in results_df.columns if col != 'filename']
results_df = results_df[cols]

output_path = f"{Subject}_cop.csv"

if not os.path.exists(output_path):
    results_df.to_csv(output_path, index=False, mode='w', header=True)
else:
    results_df.to_csv(output_path, index=False, mode='a', header=False)

## 7. Comparison Plots Across Conditions and Groups

The functions below help me compare COP ellipses across conditions (pre/post Rocker Board, pre/post Wobble Board) and groups (Adults vs Children). Centering all ellipses at the origin and using the same scale makes the spatial differences directly readable.

The hardcoded data values that originally drove these plots have been removed; pass your own ellipse parameter lists (each entry needs `label`, `width`, `height`, `angle`) to use the functions.

In [ ]:
def plot_ellipses_comparison(ellipses):
    """
    Plot multiple ellipses centered at the origin for visual comparison.

    Parameters
    ----------
    ellipses : list of dict
        Each dict has keys:
        - label  : str (e.g., "Adult preRB")
        - width  : float (major axis length)
        - height : float (minor axis length)
        - angle  : float (rotation angle in degrees)
    """
    fig, ax = plt.subplots(figsize=(8, 8))

    max_diam = 0.0

    for e in ellipses:
        width  = e["width"]
        height = e["height"]
        angle  = e["angle"]
        label  = e["label"]

        # Ellipse area for the legend
        area = np.pi * (width / 2.0) * (height / 2.0)

        ell = Ellipse(
            xy=(0.0, 0.0),      # Same center so shapes can be compared directly
            width=width,
            height=height,
            angle=angle,
            fill=False,
            lw=2,
        )
        ax.add_patch(ell)

        # Legend entry shows area alongside the label
        ax.plot([], [], label=f"{label} (A={area:.2f})")

        max_diam = max(max_diam, width, height)

    # Equal aspect so ellipse shapes are not distorted
    r = max_diam * 0.6
    ax.set_xlim(-r, r)
    ax.set_ylim(-r, r)
    ax.set_aspect("equal")

    ax.set_xlabel("X (arbitrary units)")
    ax.set_ylabel("Y (arbitrary units)")
    ax.set_title("Ellipse comparison: Adults vs Children, pre/post RB/WB")
    ax.grid(True)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


### Side-by-side Adults vs Children panel

The two-panel version uses color coding by condition so the same condition (e.g., preRB) is visually consistent across the Adults and Children panels.

In [ ]:
def plot_ellipses_adults_children(ellipses_adult, ellipses_child):
    """
    Plot ellipses for Adults and Children groups side by side.

    Parameters
    ----------
    ellipses_adult, ellipses_child : list of dict
        Each dict has keys:
        - label  : str  (e.g., "preRB", "postRB", "preWB", "postWB")
        - width  : float (major axis length)
        - height : float (minor axis length)
        - angle  : float (rotation angle in degrees)
    """

    # Color coding by condition so the same condition is consistent across both panels
    cond_colors = {
        "preRB":  "tab:blue",
        "postRB": "tab:orange",
        "preWB":  "tab:green",
        "postWB": "tab:red",
    }

    # Use a common scale derived from the largest ellipse across both groups
    max_diam = 0.0
    for e in ellipses_adult + ellipses_child:
        max_diam = max(max_diam, e["width"], e["height"])
    r = max_diam * 0.6  # a bit of padding

    fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)

    groups = [("Adults", ellipses_adult), ("Children", ellipses_child)]

    for ax, (gname, ell_list) in zip(axes, groups):
        for e in ell_list:
            width  = e["width"]
            height = e["height"]
            angle  = e["angle"]
            cond_label = e["label"]

            color = cond_colors.get(cond_label, "k")  # fallback to black for unrecognized labels

            # Ellipse area for the legend
            area = np.pi * (width / 2.0) * (height / 2.0)

            ell = Ellipse(
                xy=(0.0, 0.0),
                width=width,
                height=height,
                angle=angle,
                fill=False,
                lw=2,
                edgecolor=color,
            )
            ax.add_patch(ell)

            # Dummy line for the legend (uses color and label only)
            ax.plot([], [], color=color, label=f"{cond_label} (A={area:.2f})")

        ax.set_xlim(-r, r)
        ax.set_ylim(-r, r)
        ax.set_aspect("equal")
        ax.grid(True)
        ax.set_title(gname)
        ax.set_xlabel("X (unit)")
        ax.set_ylabel("Y (unit)")

    fig.suptitle("Ellipse comparison: Adults vs Children (pre/post RB/WB)")
    axes[1].legend(title="Condition (Area)", fontsize=8)
    plt.tight_layout()
    plt.show()
